# 🔍 Diagnostic: Tại sao DEL tăng liên tục?

Notebook đơn giản để chẩn đoán vấn đề DEL curve.

**Yêu cầu:** Bạn phải chạy notebook chính (Markovchain.ipynb) trước để có các biến:
- `matrices_by_mob`
- `parent_fallback`
- `k_final_by_mob`
- `forecast_results` (hoặc `forecast_calibrated`)
- `disb_total_by_vintage`

---

## Bước 1: Import diagnostic scripts

In [ ]:
# Import
import sys
from pathlib import Path

# Add project root to path
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("📥 Importing diagnostic scripts...")

try:
    from diagnose_why_increase_after_24 import diagnose_why_increase_after_24
    from check_p24_quality import check_p24_quality
    from diagnose_del_curve import diagnose_del_curve
    print("✅ Import thành công!")
except ImportError as e:
    print(f"❌ Lỗi import: {e}")
    print("\n💡 Đảm bảo các file này có trong project root:")
    print("   - diagnose_why_increase_after_24.py")
    print("   - check_p24_quality.py")
    print("   - diagnose_del_curve.py")

## Bước 2: Kiểm tra biến có sẵn không

In [ ]:
print("🔍 Kiểm tra biến cần thiết...\n")

# Check variables
required_vars = {
    'matrices_by_mob': 'Transition matrices',
    'parent_fallback': 'Parent fallback matrices',
    'k_final_by_mob': 'K values by MOB',
    'disb_total_by_vintage': 'Disbursement totals'
}

all_good = True
for var_name, description in required_vars.items():
    if var_name in globals():
        print(f"✅ {var_name:25s} - {description}")
    else:
        print(f"❌ {var_name:25s} - MISSING!")
        all_good = False

# Check forecast results (có thể là forecast_results hoặc forecast_calibrated)
if 'forecast_results' in globals():
    print(f"✅ {'forecast_results':25s} - Forecast data")
    forecast_data = forecast_results
elif 'forecast_calibrated' in globals():
    print(f"✅ {'forecast_calibrated':25s} - Forecast data")
    forecast_data = forecast_calibrated
else:
    print(f"❌ {'forecast_results':25s} - MISSING!")
    all_good = False
    forecast_data = None

print("\n" + "="*60)
if all_good:
    print("✅ TẤT CẢ BIẾN ĐÃ SẴN SÀNG!")
    print("   Có thể chạy diagnostic.")
else:
    print("❌ THIẾU BIẾN!")
    print("\n💡 Bạn cần:")
    print("   1. Mở notebook chính (Markovchain.ipynb)")
    print("   2. Chạy tất cả cells đến hết phần Calibration")
    print("   3. Sau đó quay lại notebook này")
print("="*60)

## Bước 3: Chạy Diagnostic

**Đây là bước quan trọng nhất!**

Chỉ chạy cell này nếu Bước 2 cho kết quả ✅ TẤT CẢ BIẾN ĐÃ SẴN SÀNG.

In [ ]:
print("🔍 CHẠY DIAGNOSTIC...")
print("="*80)

try:
    diagnose_why_increase_after_24(
        matrices_by_mob=matrices_by_mob,
        parent_fallback=parent_fallback,
        k_final_by_mob=k_final_by_mob,
        forecast_results=forecast_data,
        disb_total_by_vintage=disb_total_by_vintage,
        df_del_product=None
    )
    
    print("\n" + "="*80)
    print("✅ DIAGNOSTIC HOÀN THÀNH!")
    print("="*80)
    print("\n📝 Đọc kết quả ở trên để xác định vấn đề.")
    print("\n💡 Các giải pháp:")
    print("   - Nếu K quá cao → Chạy cell 'Giải pháp 1' bên dưới")
    print("   - Nếu nhiều fallback → Chạy cell 'Giải pháp 2' bên dưới")
    print("   - Xem thêm: HUONG_DAN_CHAY_DIAGNOSTIC.md")
    
except NameError as e:
    print(f"\n❌ Lỗi: Thiếu biến - {e}")
    print("\n💡 Chạy lại Bước 2 để kiểm tra biến nào bị thiếu.")
except Exception as e:
    print(f"\n❌ Lỗi: {e}")
    import traceback
    traceback.print_exc()

---

## Giải pháp 1: Cap K ở MOB 25+

**Chỉ chạy cell này nếu diagnostic cho thấy: ❌ K values quá cao**

In [ ]:
print("🔧 ÁP DỤNG GIẢI PHÁP 1: Cap K ở MOB 25+")
print("="*60)

print("\nK values TRƯỚC KHI CAP:")
for mob in range(24, 37):
    k_val = k_final_by_mob.get(mob, 1.0)
    status = "❌ Cao" if k_val > 0.9 else "✅ OK"
    print(f"  MOB {mob}: {k_val:.3f} {status}")

# Cap K
print("\n🔧 Đang cap K...")
for mob in range(25, 37):
    if mob in k_final_by_mob:
        k_final_by_mob[mob] = min(k_final_by_mob[mob], 0.3)
    else:
        k_final_by_mob[mob] = 0.3

print("\nK values SAU KHI CAP:")
for mob in range(24, 37):
    k_val = k_final_by_mob.get(mob, 1.0)
    print(f"  MOB {mob}: {k_val:.3f} ✅")

print("\n" + "="*60)
print("✅ ĐÃ CAP K!")
print("="*60)
print("\n💡 Bước tiếp theo:")
print("   1. Quay lại notebook chính")
print("   2. Re-run forecast với k_final_by_mob đã được cap")
print("   3. Quay lại đây chạy lại Bước 3 để verify")

## Giải pháp 2: Tăng MIN_OBS/MIN_EAD

**Chỉ áp dụng nếu diagnostic cho thấy: ❌ Nhiều cohorts dùng fallback**

In [ ]:
print("📝 GIẢI PHÁP 2: Tăng MIN_OBS/MIN_EAD")
print("="*60)
print("\n⚠️  Giải pháp này yêu cầu sửa file src/config.py")
print("\n📝 Các bước:")
print("   1. Mở file: src/config.py")
print("   2. Tìm dòng: MIN_OBS = 100")
print("   3. Sửa thành: MIN_OBS = 200")
print("   4. Tìm dòng: MIN_EAD = 1e2")
print("   5. Sửa thành: MIN_EAD = 5e2")
print("   6. Save file")
print("   7. Restart kernel và chạy lại từ đầu")
print("\n💡 Hoặc xem chi tiết trong: HUONG_DAN_CHAY_DIAGNOSTIC.md")
print("="*60)

---

## Optional: Check P_24 Quality

Kiểm tra chi tiết ma trận P_24 cho một cohort mẫu.

In [ ]:
print("🔍 KIỂM TRA P_24 QUALITY")
print("="*60)

try:
    # Get sample cohort
    if matrices_by_mob:
        sample_product = list(matrices_by_mob.keys())[0]
        
        if 24 in matrices_by_mob[sample_product]:
            sample_score = list(matrices_by_mob[sample_product][24].keys())[0]
            
            print(f"\n📊 Sample: Product={sample_product}, Score={sample_score}\n")
            
            P_24, P_parent = check_p24_quality(
                matrices_by_mob=matrices_by_mob,
                parent_fallback=parent_fallback,
                product=sample_product,
                score=sample_score
            )
        else:
            print("⚠️  MOB 24 không tồn tại")
    else:
        print("⚠️  matrices_by_mob rỗng")
        
except Exception as e:
    print(f"❌ Lỗi: {e}")

---

## ✅ HOÀN THÀNH!

### Tóm tắt:

1. ✅ Chạy diagnostic để xác định vấn đề
2. ✅ Áp dụng giải pháp phù hợp
3. ✅ Re-run forecast và verify

### Tài liệu:

- **`HUONG_DAN_CHAY_DIAGNOSTIC.md`** - Hướng dẫn đầy đủ (Tiếng Việt)
- **`NEXT_STEPS_DIAGNOSIS.md`** - Detailed guide (English)
- **`DIAGNOSIS_CONTINUOUS_INCREASE.md`** - Lý thuyết

---